# 01: Election Data Exploration

Load, inspect, and clean the commune-level election dataset. Goal: produce `df_clean` - 937 communes each with `abstention_rate`, `winning_nuance`, and `winning_bloc`.

Steps:
1. Load `commune.parquet`: already aggregated to commune level
2. Clean French decimal format and derive `abstention_rate`
3. Find the winning list per commune via `idxmax` on vote-share columns
4. Join `nuances.csv` to map nuance codes → political blocs
5. Fill gaps from `candidats_elu_fr_entier.parquet` where possible
6. Drop communes with no political attribution: 937 remain

## Commune data

In [ ]:
import pandas as pd

df = pd.read_parquet("../data/raw/commune.parquet")
print(df.shape)
print(df.dtypes)
print(df.head())

**Finding:** 1526 rows × 83 columns. Already aggregated to commune level: no need to touch `general_results.parquet` for abstention data. Join key is `Code commune` (5-char INSEE code).

Wide format: each commune has up to 5 list columns (`Nuance liste 1..5`, `Voix 1..5`, `% Voix/exprimés 1..5`). NaNs in list 3/4/5 mean the commune had fewer competing lists, not missing data.

In [ ]:
# inspect list-related columns to understand the wide format
cols = [c for c in df.columns if any(x in c for x in ['Nuance', 'Voix', 'Elu', 'Libellé de liste'])]
print(df[cols].head(3).to_string())

# check the raw string format of % Abstentions and whether there are nulls
print(df['% Abstentions'].head(10))
print(df['% Abstentions'].dtype)
print(df['% Abstentions'].isna().sum())

# check nulls across the 5 vote-share columns
# NaN in column N means fewer than N lists stood in that commune
voix_cols = [f'% Voix/exprimés {i}' for i in range(1, 6)]
print(df[voix_cols].isna().sum())

**Finding:** `% Abstentions` and `% Voix/exprimés` are strings in French decimal format (`"56,35%"`). Must strip `%` and replace `,` with `.` before any arithmetic.

List distribution across 1526 communes: 556 had 2 lists, 800 had 3, 154 had 4, 16 had 5.

In [ ]:
# strip % and convert French decimal comma to dot, then cast to float
df['abstention_rate'] = df['% Abstentions'].str.replace('%', '').str.replace(',', '.').astype(float)

# same cleaning for all 5 vote-share columns: these drive the winner detection below
voix_cols = [f'% Voix/exprimés {i}' for i in range(1, 6)]
for col in voix_cols:
    df[col] = df[col].str.replace('%', '').str.replace(',', '.').astype(float)

print(df['abstention_rate'].describe())
print(df[voix_cols].describe())

**Sanity check:** abstention rates range 0.94%–64.41%, mean 35.2%. Two communes under 2%: Ambricourt (106 registered) and Aix-la-Fayette (114 registered). Small villages with near-total turnout. Data is correct.

In [ ]:
print(df[df['abstention_rate'] < 2])

## Winning list per commune

`idxmax()` across the five `% Voix/exprimés` columns returns the column name with the highest vote share per row. From that column name we extract the list number (1–5) and pull the matching `Nuance liste X` value.

We do not use `Elu`: in French municipal elections `Elu` counts council seats won proportionally across multiple lists. `% Voix/exprimés` is the correct measure of the dominant list.

In [ ]:
# idxmax returns the column label with the max value per row
# e.g. '% Voix/exprimés 3' → the third list won in that commune
winning_col = df[voix_cols].idxmax(axis=1)

# extract the list number from the column name: last token after split on space
winning_num = winning_col.str.split(' ').str[-1]

print(winning_col.head())
print(winning_num.head())

# use the extracted number to look up 'Nuance liste X' on the same row
df['winning_nuance'] = [
    row[f'Nuance liste {num}']
    for (_, row), num in zip(df.iterrows(), winning_num)
]

print(df[['Libellé commune', 'winning_nuance']].head())

In [ ]:
# nuances.csv maps nuance codes (e.g. 'LREN') to political blocs (e.g. 'CENT')
nuances = pd.read_csv('../data/raw/nuances.csv', sep=';')

# municipal elections use lists: filter out individual-candidate nuances
nuances_liste = nuances[nuances['type_nuance'] == 'liste']

# left join: keep all df rows; unmatched → NaN winning_bloc
# rename bloc before merge to avoid collision if cell is re-run
df = df.merge(
    nuances_liste[['libelle_nuance', 'bloc']].rename(columns={'bloc': 'winning_bloc'}),
    left_on='winning_nuance',
    right_on='libelle_nuance',
    how='left'
)

print(df[['Libellé commune', 'winning_nuance', 'winning_bloc']].head())
print('unmatched blocs:', df['winning_bloc'].isna().sum())

## Political bloc attribution: data gap

**Finding:** 589 of 1526 communes have no `Nuance liste` codes in `commune.parquet`: all list columns are NaN despite having valid vote percentages. These are not data errors; the source file has no nuance attribution for these communes.

**Investigation:**
- All 589 have valid `% Voix/exprimés` data
- Spread across many departments, not a regional gap
- Mean registered voters: 1275 (vs 17212 for complete communes)
- Mean abstention: 26% (vs 41% for complete communes)
- `candidats_elu_fr_entier.parquet` partially covers this gap: see fill step below

**Decision:** After filling from the `elu` file, 937 communes retain complete political + abstention data. Limitation noted: smaller communes with lower abstention are underrepresented in the final dataset.

In [ ]:
# inspect what nuance values are present among the unmatched rows
unmatched = df[df['winning_bloc'].isna()]['winning_nuance'].value_counts()
print(unmatched)

# check how many have no nuance at all (vs nuance present but bloc lookup failed)
print(df['winning_nuance'].isna().sum())

# spot-check the communes with no nuance: what does their raw data look like?
print(df[df['winning_nuance'].isna()][['Libellé commune', 'Nuance liste 1', 'Nuance liste 2', 'Nuance liste 3', '% Voix/exprimés 1', '% Voix/exprimés 2', '% Voix/exprimés 3']].head(10))

# verify the gap is not concentrated in a single department
print(df[df['winning_nuance'].isna()]['Code département'].value_counts().head(15))

In [ ]:
elu = pd.read_parquet('../data/raw/candidats_elu_fr_entier.parquet')
print(elu.shape)
print(elu.columns.tolist())
print(elu.head(3))
print(elu['CODCOM'].nunique())
print(elu['TOUR_ELECTION'].value_counts())

# derive one nuance per commune from the elu file:
# take the modal nuance code among elected candidates: handles multi-list councils
nuance_from_elu = (
    elu.groupby('CODCOM')['CODE_NUANCE_DE_LISTE']
    .agg(lambda x: x.dropna().mode().iloc[0] if not x.dropna().empty else None)
    .reset_index()
    .rename(columns={'CODE_NUANCE_DE_LISTE': 'nuance_from_elu'})
)

print(nuance_from_elu.head())
print(nuance_from_elu.shape)

In [ ]:
# join elu-derived nuances: left join keeps all df rows
df = df.merge(nuance_from_elu, left_on='Code commune', right_on='CODCOM', how='left')

# fill missing winning_nuance only where the elu file has a value
df['winning_nuance'] = df['winning_nuance'].fillna(df['nuance_from_elu'])

# re-derive winning_bloc for rows that were just filled
# uses a Series map rather than a second merge to avoid adding columns
df['winning_bloc'] = df['winning_bloc'].fillna(
    df['winning_nuance'].map(nuances_liste.set_index('libelle_nuance')['bloc'])
)

print('still missing nuance:', df['winning_nuance'].isna().sum())
print('still missing bloc:', df['winning_bloc'].isna().sum())
print(df[['Libellé commune', 'winning_nuance', 'winning_bloc']].head(10))

In [ ]:
# drop the 589 communes with no recoverable political attribution
df_clean = df[df['winning_bloc'].notna()].copy()
print(df_clean.shape)
print(df_clean['winning_bloc'].value_counts())
df_clean.head()

## Election data: summary

`df_clean` contains 937 communes, each with complete political and abstention data.

| Column | Description |
| --- | --- |
| `Code commune` | 5-char INSEE code: join key for BPE and population data |
| `abstention_rate` | Float, % of registered voters who did not vote |
| `winning_nuance` | Nuance code of the list with the highest `% Voix/exprimés` |
| `winning_bloc` | Political bloc: EXG / GAU / CENT / DTE / EXD / DIV |

**Next:** Load `BPE24.parquet`, filter to transport equipment codes, and aggregate to commune level to derive `transport_score`.

In [ ]:
import json
import os

# select and rename columns for the export: snake_case throughout
export_df = df_clean[['Code commune', 'Libellé commune', 'abstention_rate', 'winning_bloc', 'winning_nuance']].rename(columns={
    'Code commune': 'code_commune',
    'Libellé commune': 'nom_commune',
})

# write JSON: records orient produces a list of dicts, one per commune
out_path = '../data/processed/elections-base.json'
os.makedirs(os.path.dirname(out_path), exist_ok=True)

export_df.to_json(out_path, orient='records', force_ascii=False, indent=2)

file_size_kb = os.path.getsize(out_path) / 1024
print(f"Exported {len(export_df)} rows → {out_path}")
print(f"File size: {file_size_kb:.1f} KB")